# YOGA Chatbot — Training Pipeline (Hybrid Intent Classifier)

**YOgyakarta Guide Assistant** — 3-stage hybrid SVM intent classifier.

This notebook documents the **reproducible** training methodology. It mirrors
`scripts/train.py` and deliberately avoids three mistakes present in the
original notebook:

1. **No data leakage** — the train/test split happens *before* augmentation
   and the TF-IDF vectorizers are fit on the **training split only**.
2. **Train/inference parity** — patterns are preprocessed with the *exact*
   runtime transform (`EntityExtractor` location placeholder `[LOKASI]` +
   Sastrawi stemming).
3. **Semantic intents** — 12 semantic classes the bot actually routes on;
   location is handled separately by the `EntityExtractor`.

All metrics below are reported on an **untouched held-out test split**.

## 1. Imports

In [1]:
import sys
import random
import json
import pickle
import collections
from pathlib import Path
from datetime import date

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from yoga_chatbot.nlu.entity_extractor import EntityExtractor
from yoga_chatbot.nlu.intent_classifier import GREETING_INTENTS
from yoga_chatbot.preprocessing.text_processor import TextProcessor
from augment_data import augment_pattern

sns.set_style("whitegrid")
random.seed(42)
np.random.seed(42)
RANDOM_STATE = 42
print("sklearn", sklearn.__version__, "| numpy", np.__version__)

sklearn 1.4.2 | numpy 1.26.4


## 2. Load the semantic intent dataset

`data/raw/intents_semantic.json` is produced by `scripts/relabel_intents.py`,
which maps the raw location-tagged patterns onto the 12 semantic intents.

In [2]:
with open(ROOT / "data/raw/intents_semantic.json", encoding="utf-8") as f:
    data = json.load(f)

patterns, labels = [], []
for intent in data["intents"]:
    for p in intent["patterns"]:
        patterns.append(p)
        labels.append(intent["tag"])

print(f"Total patterns: {len(patterns)}")
print(f"Classes ({len(set(labels))}):")
for tag, n in collections.Counter(labels).most_common():
    print(f"  {tag:20} {n}")

Total patterns: 2272
Classes (12):
  rekomendasi_wisata   1608
  cari_by_rating       185
  info_lokasi          177
  info_detail          76
  cari_by_type         61
  greeting             25
  pagi                 25
  siang                25
  sore                 25
  malam                25
  goodbye              25
  cari_by_harga        15


## 3. Runtime-parity preprocessing

At serving time the pipeline replaces the location with `[LOKASI]` and then
stems the text. We apply the identical transform here so training matches
inference.

In [3]:
extractor = EntityExtractor(ROOT / "data/knowledge/kecamatan_diy.json")
processor = TextProcessor()

def preprocess(raw: str) -> str:
    neutral = extractor.replace_with_placeholder(raw)
    return processor.preprocess(neutral)

for example in ["rekomendasi pantai di bantul", "tiket murah 30rb", "selamat pagi"]:
    print(f"{example!r:35} -> {preprocess(example)!r}")

'rekomendasi pantai di bantul'      -> 'rekomendasi pantai di lokasi'
'tiket murah 30rb'                  -> 'tiket murah 30rb'
'selamat pagi'                      -> 'selamat pagi'


## 4. Train/test split — BEFORE augmentation

Splitting first guarantees that augmented variants of a training sentence can
never leak into the test set. The split is stratified by intent.

In [4]:
p_train, p_test, y_train_lbl, y_test_lbl = train_test_split(
    patterns, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels,
)
print(f"Train (pre-aug): {len(p_train)}   Test: {len(p_test)}")

Train (pre-aug): 1817   Test: 455


## 5. Augment the training split only

`augment_pattern` applies intent-aware augmentation. The test split stays raw.

In [5]:
aug_patterns, aug_labels = [], []
for pattern, label in zip(p_train, y_train_lbl):
    for variant in augment_pattern(pattern, label):
        aug_patterns.append(variant)
        aug_labels.append(label)

print(f"Train after augmentation: {len(aug_patterns)} "
      f"(x{len(aug_patterns)/len(p_train):.1f})")

X_train_text = [preprocess(p) for p in aug_patterns]
X_test_text  = [preprocess(p) for p in p_test]

label_encoder = LabelEncoder().fit(aug_labels + y_test_lbl)
y_train = label_encoder.transform(aug_labels)
y_test  = label_encoder.transform(y_test_lbl)

Train after augmentation: 6946 (x3.8)

## 6. Feature extraction — dual TF-IDF fit on TRAIN only

`fit_transform` is called on training text; test text is only `transform`ed.
No test information enters the vocabulary or IDF weights.

In [6]:
tfidf_main = TfidfVectorizer(
    max_features=2000, ngram_range=(1, 2), min_df=2, max_df=0.9,
    token_pattern=r"\b\w+\b",
)
X_train_main = tfidf_main.fit_transform(X_train_text)
X_test_main  = tfidf_main.transform(X_test_text)

tfidf_greeting = TfidfVectorizer(
    max_features=500, ngram_range=(1, 2), min_df=1, max_df=0.9,
    token_pattern=r"\b\w+\b",
)
X_train_greet = tfidf_greeting.fit_transform(X_train_text)

print(f"Main TF-IDF:     {X_train_main.shape}")
print(f"Greeting TF-IDF: {X_train_greet.shape}")

Main TF-IDF:     (6946, 1619)
Greeting TF-IDF: (6946, 500)


## 7. Stage 1 — binary GreetingDetector

A balanced linear SVM separating greetings from everything else; the hybrid
pipeline uses it as a fast gate for short inputs.

In [7]:
greeting_codes = set(label_encoder.transform(sorted(GREETING_INTENTS)))
y_train_bin = np.array([1 if c in greeting_codes else 0 for c in y_train])

greeting_detector = SVC(kernel="linear", C=1.0, probability=True,
                        class_weight="balanced", random_state=RANDOM_STATE)
greeting_detector.fit(X_train_greet, y_train_bin)
print("GreetingDetector trained "
      f"(greeting={int(y_train_bin.sum())}, non-greeting={int((y_train_bin==0).sum())})")

GreetingDetector trained (greeting=1087, non-greeting=5859)


## 8. Stage 2 — main semantic classifier

A balanced 12-class linear SVM with probability estimates (needed for the
confidence-based fallback).

In [8]:
main_svm = SVC(kernel="linear", C=1.0, probability=True,
               class_weight="balanced", random_state=RANDOM_STATE)
main_svm.fit(X_train_main, y_train)

y_pred = main_svm.predict(X_test_main)
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Held-out test accuracy: {acc*100:.2f}%")
print(f"Macro F1-score:         {macro_f1*100:.2f}%")

Held-out test accuracy: 94.73%
Macro F1-score:         88.80%


## 9. Classification report (held-out test)

In [9]:
print(classification_report(
    y_test, y_pred, target_names=label_encoder.classes_,
    digits=3, zero_division=0,
))

                    precision    recall  f1-score   support

     cari_by_harga      1.000     1.000     1.000         3
    cari_by_rating      1.000     0.973     0.986        37
      cari_by_type      0.750     0.750     0.750        12
           goodbye      1.000     0.600     0.750         5
          greeting      0.667     0.800     0.727         5
       info_detail      0.600     1.000     0.750        15
       info_lokasi      0.921     0.972     0.946        36
             malam      1.000     1.000     1.000         5
              pagi      1.000     0.800     0.889         5
rekomendasi_wisata      0.981     0.957     0.969       322
             siang      1.000     1.000     1.000         5
              sore      1.000     0.800     0.889         5

          accuracy                          0.947       455
         macro avg      0.910     0.888     0.888       455
      weighted avg      0.957     0.947     0.949       455



## 10. Confusion matrix

In [10]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — 12 Semantic Intents (held-out test)")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23572\3204210820.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 11. Per-class F1-score

In [11]:
prec, rec, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=range(len(label_encoder.classes_)), zero_division=0,
)
order = np.argsort(f1)
plt.figure(figsize=(10, 6))
colors = ["green" if v >= 0.8 else "orange" if v >= 0.5 else "red" for v in f1[order]]
plt.barh(range(len(order)), f1[order], color=colors, alpha=0.8, edgecolor="black")
plt.yticks(range(len(order)), [label_encoder.classes_[i] for i in order])
plt.axvline(0.8, color="gray", ls="--", lw=1)
plt.xlabel("F1-score"); plt.xlim(0, 1.0)
plt.title("Per-class F1 (held-out test)")
plt.tight_layout(); plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23572\2203093383.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 12. Persist artifacts

Save the five fully-fitted artifacts the bot loads, plus `metadata.json`.
In production prefer `scripts/train.py`, which performs these exact steps
non-interactively.

In [12]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

def dump(obj, name):
    with open(MODEL_DIR / name, "wb") as f:
        pickle.dump(obj, f)

dump(tfidf_main,        "tfidf_vectorizer.pickle")
dump(tfidf_greeting,    "tfidf_greeting.pickle")
dump(main_svm,          "svm_model.pkl")
dump(greeting_detector, "greeting_detector.pkl")
dump(label_encoder,     "label_encoder.pickle")

metadata = {
    "trained_on": str(date.today()),
    "input_dataset": "data/raw/intents_semantic.json",
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "classes": list(label_encoder.classes_),
    "n_classes": len(label_encoder.classes_),
    "train_samples_pre_aug": len(p_train),
    "train_samples_post_aug": len(aug_patterns),
    "test_samples": len(p_test),
    "test_accuracy": round(float(acc), 4),
    "test_macro_f1": round(float(macro_f1), 4),
    "random_state": RANDOM_STATE,
}
with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("Saved artifacts + metadata to", MODEL_DIR)

Saved artifacts + metadata to C:\Users\LENOVO\code\YOGA-Chatbot\models


## 13. Smoke test — full hybrid pipeline

Load the saved artifacts through the full `NLUPipeline` to confirm end-to-end
behaviour (confidence fallback + entity extraction).

In [13]:
from yoga_chatbot.nlu.pipeline import NLUPipeline

class _S:
    model_dir = ROOT / "models"
    kecamatan_path = ROOT / "data/knowledge/kecamatan_diy.json"
    greeting_confidence_threshold = 0.7
    word_count_threshold = 3
    intent_confidence_threshold = 0.15

nlu = NLUPipeline.from_settings(_S)
tests = [
    "halo", "selamat pagi", "wisata di bantul", "pantai di gunungkidul",
    "tiket murah 30rb", "rating terbaik", "info candi prambanan",
    "lokasi pantai parangtritis", "asdkjh qweqwe zxcvb",
]
for t in tests:
    r = nlu.understand(t)
    ent = f"{r.entity['type']}:{r.entity['value']}" if r.entity["value"] else "-"
    print(f"{t:32} -> {r.intent:20} ({r.confidence:.2f})  entity={ent}")

halo                             -> greeting             (0.62)  entity=-
selamat pagi                     -> pagi                 (0.25)  entity=-
wisata di bantul                 -> rekomendasi_wisata   (0.93)  entity=kecamatan:bantul
pantai di gunungkidul            -> cari_by_type         (0.99)  entity=kabupaten:gunungkidul
tiket murah 30rb                 -> cari_by_harga        (0.96)  entity=-
rating terbaik                   -> rekomendasi_wisata   (0.84)  entity=-
info candi prambanan             -> info_detail          (0.45)  entity=kecamatan:prambanan
lokasi pantai parangtritis       -> cari_by_type         (1.00)  entity=-
asdkjh qweqwe zxcvb              -> rekomendasi_wisata   (0.84)  entity=-
